![Databricks Academy](./Includes/images/common/db-academy.png)

# 07 - Adding Data Quality Expectations

In this demonstration, we will add data quality expectations to apply quality constraints that validate data as it flows through Apache Spark™ Declarative Pipelines. Expectations provide greater insight into data quality metrics and allow you to fail updates or drop records when detecting invalid records.


### Learning Objectives

By the end of this lesson, you will be able to:
- Add quality constraints within a Apache Spark™ Declarative Pipeline to trigger appropriate actions (warn, drop, or fail) based on data expectations.
- Analyze pipeline metrics to identify and interpret data quality issues across different data flows.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the following cell to configure your working environment for this course.

    This cell will also reset your `/Volumes/learn_build_data_pipelines_sdp_YOUR_USERNAME/sdp_1_bronze/source` volume with the JSON files to the starting point, with one JSON file in each directory.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-49569909-813c-47eb-96c1-2d55788cda93
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-49569909-813c-47eb-96c1-2d55788cda93
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.

ℹ️ Non-Vocareum workspace detected. Setting up catalog.
ℹ️ Using default catalog name: 'learn_build_data_pipelines_sdp_joemar_brillo'.
✅ Catalog 'learn_build_data_pipelines_sdp_joemar_brillo' already exists. Using this catalog.



  STEP 1: Verifying catalog exists: learn_build_data_pipelines_sdp_joemar_brillo
  Catalog 'learn_build_data_pipelines_sdp_joemar_brillo' exists.

  STEP 2: Setting up 3 schema(s) in catalog: learn_build_data_pipelines_sdp_joemar_brillo
  [1/3] Checking: `learn_build_data_pipelines_sdp_joemar_brillo`.`sdp_1_bronze`... ALREADY EXISTS
  [2/3] Checking: `learn_build_data_pipelines_sdp_joemar_brillo`.`sdp_2_silver`... ALREADY EXISTS
  [3/3] Checking: `learn_build_data_pipelines_sdp_joemar_brillo`.`sdp_3_gold`... ALREADY EXISTS

  COMPLETE: 0 schema(s) created, 3 already existed.



DataFrame[]


  STEP 1: Validating volume folder path...
  Found: /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/customers

  STEP 2: Scanning for files...
  Found 1 file(s) to delete.

  STEP 3: Deleting files from: /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/customers
  [1/1] Deleting: 00.json... DELETED

  COMPLETE: Deleted 1 file(s) from /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/customers


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/joemar.brillo@outlook.com/Build Data Pipelines with Apache Spark Declarative Pipelines/Includes/data/customers

  STEP 2: Checking target volume path...
  Target volume path already exists: /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/customers

  STEP 3: Reading source files...
  Found 5 file(s) in source folder.

  STEP 4: Copying 1 file(s) to target volume...
  Source:      /Workspace/Users/joemar.brill

Information,Value
Your Catalog:,learn_build_data_pipelines_sdp_joemar_brillo
Bronze Schema:,sdp_1_bronze
Silver Schema:,sdp_2_silver
Gold Schema:,sdp_3_gold
Source Volume:,/Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source


Compute,Status,Details
Serverless,✓ Match,Version 5


2. Run the cell below to programmatically view the files in your `/Volumes/learn_build_data_pipelines_sdp_YOUR_USERNAME/sdp_1_bronze/source/orders/` volume.

    Confirm you only see the original **00.json** file in the **orders** folder.

In [0]:
%python
spark.sql(f'LIST "{source_volume_path}/orders"').display()

path,name,size,modification_time
/Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source/orders/00.json,00.json,15313,1794021881000


## B. Adding Data Quality Expectations

This demonstration includes the simple starter Spark Declarative Pipeline that has already been created in the previous demonstration.

  We will continue to build on it to explore its capabilities.

  **Manage data quality with pipeline expectations**:
[AWS](https://docs.databricks.com/aws/en/ldp/expectations) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/expectations) |
[GCP](https://docs.databricks.com/gcp/en/ldp/expectations)


1. Run the cell below to create your starter pipeline for this demonstration (pipeline from the previous demonstration).

    The setup code will set the following for you:

    - Your default catalog: **learn_build_data_pipelines_sdp_YOUR_USERNAME**

    - Your configuration parameter: `source` = `/Volumes/learn_build_data_pipelines_sdp_YOUR_USERNAME/sdp_1_bronze/source/`

      **NOTE:** If the pipeline already exists, an error will be returned. In that case, you'll need to delete the existing pipeline and rerun this cell.

      To delete the pipeline:

      - Select **Jobs and Pipelines** from the far-left navigation bar.

      - Find the pipeline you want to delete.

      - Click the three-dot menu ![ellipsis icon](./Includes/images/common/ellipsis_icon.png).

      - Select **Delete**.

**NOTE:**  The `create_declarative_pipeline` function is a custom function built for this course to create the sample pipeline using the Databricks REST API. This avoids manually creating the pipeline and referencing the pipeline assets.

In [0]:
%python
create_declarative_pipeline(
    pipeline_name=f'07 - Adding Data Quality Expectations Project - {my_catalog}',
    root_path_folder_name='07 - Adding Data Quality Expectations Project',
    catalog_name=my_catalog,
    schema_name='default',
    source_folder_names=['orders'],
    configuration={'source': source_volume_path}
)


  STEP 1: Checking for existing pipeline...
  ✅ No existing pipeline named '07 - Adding Data Quality Expectations Project - learn_build_data_pipelines_sdp_joemar_brillo' found.

  STEP 2: Building pipeline configuration...
  Pipeline Name:    07 - Adding Data Quality Expectations Project - learn_build_data_pipelines_sdp_joemar_brillo
  Catalog:          learn_build_data_pipelines_sdp_joemar_brillo
  Schema:           default
  Root Path:        /Workspace/Users/joemar.brillo@outlook.com/Build Data Pipelines with Apache Spark Declarative Pipelines/07 - Adding Data Quality Expectations Project
  Source Folders:    orders
  Serverless:       True
  Photon:           True
  Channel:          CURRENT
  Continuous:       False
  Development Mode: True
  Configuration:
    source = /Volumes/learn_build_data_pipelines_sdp_joemar_brillo/sdp_1_bronze/source

  STEP 3: Creating pipeline via API...
  ✅ Pipeline '07 - Adding Data Quality Expectations Project - learn_build_data_pipelines_sdp_joemar

2. Complete the following steps to open the starter Spark Declarative Pipeline project for this demonstration:

   a. In the main navigation bar, right-click on **Jobs & Pipelines** and select **Open Link in New Tab**.

   b. In **Jobs & Pipelines** select your **07 - Adding Data Quality Expectations Project - learn_build_data_pipelines_sdp_YOUR_USERNAME** pipeline.

   c. In the **Pipeline details** pane on the far right, select **Open in Editor** (field to the right of **Source code**) to open the pipeline in the **Lakeflow Pipeline Editor**.

   d. In the new tab:
      - Select the **orders** folder (The main folder also contains the extra **python_excluded** folder that contains the Python version)

      - Click on **orders_pipeline.sql**.

## C. Explore and Run the `orders_pipeline.sql` Pipeline with Data Quality Expectations

1. Click the **Run pipeline** button to start the pipeline. When prompted to confirm the Catalog and Schema, review and click **Run pipeline** again to proceed.

    *While the pipeline is running, proceed to step 2.*

2. While the pipeline is executing:

   a. Examine `CREATE OR REFRESH STREAMING TABLE 2_silver_db.orders_silver_demo07` (Section **Bronze -> Silver (Contains Data Quality Expectations)**)

   b. Notice that it includes **3 data quality expectations** applied as data is ingested into **orders_silver_demo07**:

      | Constraint | Rule | Action |
      |------------|------|--------|
      | `valid_notifications` | `notifications` must be `'Y'` or `'N'` | **Warn** — rows are kept, violation is logged |
      | `valid_date` | `order_timestamp` must be after `'2021-12-26'` | **Drop Row** — invalid rows are removed |
      | `valid_id` | `customer_id` must not be `NULL` | **Fail Update** — pipeline fails if violated |

      <br></br>
      - **notifications** column only contains `Y` or `N` values - so `N` values will trigger a warning
      - **order_timestamp** column contains dates on `2021-12-25` - so those rows will be dropped
      - **customer_id** has no nulls in this demo - so the pipeline will pass


## D. Explore the Pipeline Run

1. After the pipeline completes, explore the **Pipeline graph** on the right.

   - It creates the:
      - **orders_bronze_demo07** > **orders_silver_demo07** > **gold_orders_by_date_demo07** pipeline

   - Notice:
      - **174 rows** were read into the bronze table
      - Only **148 rows** were read into the silver table (the table with constraints)

2. In the bottom window, make sure you are in the **Tables** tab.

   - Select **orders_silver_demo07**, then select **Table metrics**

   - Note the following in the table:

     | Metric | Value | Description |
     |--------|-------|-------------|
     | **Output records** | 148 | Rows that passed all expectations and were written to the table |
     | **Expectations** | 1 met \| 2 unmet | Total data quality expectations set on the streaming table |
     | **Dropped** | 26 | Rows that failed the `DROP ROW` expectation (14.9% failure rate) |
     | **Warnings** | 32 | Rows that failed the `WARN` expectation (14.9% failure rate) |

   - Select the link in the **Expectations** column to view detailed breakdown:

     | Constraint | Action | Failure Rate | Failed Rows |
     |------------|--------|--------------|-------------|
     | `valid_notifications` | **Allow** (warn) | 22.4% | 39 |
     | `valid_date` | **Drop** | 14.9% | 26 |


- **NOTES:**
  - If the `WARN` counts differ between the table and the popup, it's because the table view de-duplicates overlapping rows, while the popup counts failures per expectation, even when the same row fails multiple expectations.
  - Expectation metrics in the UI are per update run. To analyze data quality across multiple runs, query the pipeline event log (covered later in the course).

#### Checkpoint
![](./Includes/images/data-quality-expectations/quality-expectations-run.png)

## Additional Resources

- Manage data quality with pipeline expectations:
[AWS](https://docs.databricks.com/aws/en/dlt/expectations) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/expectations) |
[GCP](https://docs.databricks.com/gcp/en/dlt/expectations)

- Expectation recommendations and advanced patterns:
[AWS](https://docs.databricks.com/aws/en/dlt/expectation-patterns) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/expectation-patterns) |
[GCP](https://docs.databricks.com/gcp/en/dlt/expectation-patterns)

- [Data Quality Management With Databricks](https://www.databricks.com/discover/pages/data-quality-management)


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>
